# 6. TensorRT build and inference

Build real engines on the target GPU, then run crop-to-text inference. Standard split graphs permit FP16 encoder/decoder and retain FP32 refinement. Fast Tiny builds one engine with FP16 allowed in the encoder and FP32-constrained decoder/refinement, plus optional CUDA graph replay.

## Model selection

Read [0_model_options.ipynb](0_model_options.ipynb) for Base, Tiny and the custom Fast Tiny recipe. Choose `configs/alpr_base.yaml`, `configs/alpr_tiny.yaml`, or `configs/alpr_fast_tiny.yaml` via `CONFIG_PATH`. Keep that choice consistent across notebooks and use separate output folders.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "parseq_ocr").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "parseq_ocr").is_dir():
    raise RuntimeError("Open this notebook from the repository or notebooks directory")
sys.path.insert(0, str(ROOT))
from parseq_ocr.config import load_config
# Dummy presets: configs/dummy_base.yaml, dummy_tiny.yaml, dummy_fast_tiny.yaml.
# Real-data presets: configs/alpr_base.yaml, alpr_tiny.yaml, alpr_fast_tiny.yaml.
# PARSEQ_CONFIG overrides this default when it is set.
CONFIG_PATH = os.environ.get("PARSEQ_CONFIG", "configs/dummy_tiny.yaml")
config = load_config(CONFIG_PATH)
print("Config:", CONFIG_PATH)
print("Model:", config["model"], "input:", config["image_size"])
print("Artifacts:", config["output_dir"])


In [ ]:
import torch
from parseq_ocr.tensorrt import build_engines, TensorRTRecognizer

if not torch.cuda.is_available():
    raise RuntimeError("TensorRT requires a CUDA GPU")
MAX_BATCH = 32  # Reduce for a smaller GPU.
deployment = config["output_dir"] / "deployment"
engine_info = build_engines(deployment, max_batch=MAX_BATCH)
print(engine_info)

In [ ]:
from PIL import Image
from parseq_ocr.data import scan_pairs

recognizer = TensorRTRecognizer(deployment)
row = scan_pairs(config["dataset_root"] / "val", config["normalization"])[0]
with Image.open(row["path"]) as image:
    prediction = recognizer.infer([image.convert("RGB")])[0]
print({"ground_truth": row["label"], "prediction": prediction})

Engines are GPU/runtime-specific and are not committed. The build cache records the ONNX hash, GPU, precision, runtime version and batch limit. To deliberately rebuild a changed graph, use `build_engines(..., force=True)`. Pass batches no larger than the compiled limit.